# 验证选中的行业：训练、验证与最终测试回测

先运行 `etf_pca_grid_search.ipynb`。本文件只读取 `data/selected_sector/etf.csv` 和 `params.json`，沿用训练综合得分选出的参数、验证综合得分选出的行业和已保存的时间划分。这里不搜索超参数、不更换行业。

三段账户各自从空仓、净值 1 开始，均按对数收益建模、简单收益记账、固定初始资金加法累计。训练和验证用于回看选择过程，测试用于最终评估；测试结果不回流到行业排名。

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
import json

# 图中行业名使用中文字体
plt.rcParams["font.sans-serif"] = ["Arial Unicode MS", "DejaVu Sans"]
plt.rcParams["axes.unicode_minus"] = False

# 日期和超参数的来源统一为网格文件；需要修改时回到网格顶部改动并从头运行
selected_dir = Path('data/selected_sector')
params = json.loads((selected_dir / 'params.json').read_text(encoding='utf-8'))
sector = params['sector']
window, k = params['window'], params['k']
s_open, s_close = params['s_open'], params['s_close']
fee = params['fee']
hl_min, hl_max = params['half_life_min'], params['half_life_max']
train_start, train_end = pd.Timestamp(params['train_start']), pd.Timestamp(params['train_end'])
valid_start, valid_end = pd.Timestamp(params['valid_start']), pd.Timestamp(params['valid_end'])
test_start, test_end = pd.Timestamp(params['test_start']), pd.Timestamp(params['test_end'])
train_backtest_start = pd.Timestamp(params['train_backtest_start'])
output_dir = Path('outputs/selected_sector') / sector
output_dir.mkdir(parents=True, exist_ok=True)
params

## 1. 读取已经选好的 ETF 数据

名单由训练期筛选后保存，不根据验证或测试期的完整性重新筛选，不补缺失收益为 0。

In [ ]:
data = pd.read_csv(selected_dir / 'etf.csv', dtype={'code': str}, parse_dates=['date'])
codes = params['eligible_codes']
dates = pd.DatetimeIndex(data['date'].unique()).sort_values()
df_selected = data.drop_duplicates('code').set_index('code').loc[codes, ['name', 'fund_type']]
df_selected.to_csv(output_dir / 'etfs.csv', index_label='code')
print(f'验证选中的行业：{sector}，使用 {len(codes)} 只 ETF')
df_selected

## 2. 分别计算训练、验证、测试的滚动信号与回测

训练从 train_start 开始更新信号状态，但收益从网格保存的共同训练回测起点开始，因此能与网格结果对应。验证和测试各自向前取 window 个交易日预热，只在对应日期区间记收益，不继承上一段持仓。

以下保留原 PCA、AR(1)、s 信号、方向与权重公式。半衰期沿用保存的范围。信号和权重先写 CSV，回测再读取；不是用测试结果选择参数。

In [ ]:
backtests, curves, metrics = {}, {}, {}
periods = {'train': (train_start, train_end), 'valid': (valid_start, valid_end), 'test': (test_start, test_end)}
for phase, (phase_start, phase_end) in periods.items():
    if phase == 'train':
        phase_data = data.loc[data['date'].between(train_start, train_end)]
        start = train_backtest_start
    else:
        i = dates.searchsorted(phase_start)
        start = dates[i]
        # 第一期信号需要之前 window 个交易日，预热期收益不计入本段
        phase_data = data.loc[data['date'].between(dates[i-window], phase_end)]
    returns = phase_data.pivot(index='date', columns='code', values='log_ret_1d')[codes]
    scores = pd.DataFrame(np.nan, index=returns.index, columns=returns.columns)
    half_lives, vols = scores.copy(), scores.copy()
    loadings = {}
    for i in range(window - 1, len(returns)):
        sample = returns.iloc[i-window+1:i+1].to_numpy() # shape(w, n),n是ETF数量
        std = sample.std(axis=0)# 对windows内取波动率
        standardized = (sample - sample.mean(axis=0)) / std # 归一化，方便用主成分
        _, _, vt = np.linalg.svd(standardized, full_matrices=False) # 得到的是vt矩阵，shape(n, n)
        loading = vt[:k] # k是主成分数量 shape(k, n)
        residual = standardized - (standardized @ loading.T) @ loading # 先将原来的n个etf收益转换为k个主成分因子，然后再利用主成分重构原始数据，最后计算残差，shape(w,n)
        # 这里的残差是剥除掉波动率最高的k个主成分因子后的残差，得到的是每个etf的各自的特点
        X = residual.cumsum(axis=0) # 按照window内对残差求和，shape(n)
        lag, current = X[:-1], X[1:] # 取前w-1个etf的残差，当前etf的残差
        centered = lag - lag.mean(axis=0)
        # 拟合ar(1)方程 x_t = a + b * x_t-1 + e_t
        b = (centered * (current - current.mean(axis=0))).sum(axis=0) / (centered**2).sum(axis=0) # 直接计算一阶滞后相关系数
        a = current.mean(axis=0) - b * lag.mean(axis=0)
        innovation = current - a - b * lag
        b = np.where((b > 0) & (b < 1), b, np.nan)
        m = a / (1 - b) # 假设x_t-1的期望= x_t的期望，也就是期望不变的前提下
        sigma_eq = innovation.std(axis=0, ddof=2) / np.sqrt(1 - b**2) # 当前累计残差偏离均值多少个标准差
        date = returns.index[i]
        scores.loc[date] = (X[-1] - m) / sigma_eq # 计算z-score
        half_lives.loc[date] = -np.log(2) / np.log(b)
        vols.loc[date] = std
        loadings[date] = loading
    # 先建日期 × ETF 的权重表，初始全部空仓
    weights = pd.DataFrame(0.0, index=scores.index, columns=scores.columns)
    # 记录每只 ETF 的残差交易方向：1 做多，-1 做空，0 空仓
    direction = np.zeros(len(scores.columns))
    # 按日期更新方向，未触发开平仓条件时保留原方向
    for date, loading in loadings.items():
        # 取当天每只 ETF 的 s 信号
        s = scores.loc[date].to_numpy()
        # 取当天估计的均值回归半衰期，单位是交易日
        hl = half_lives.loc[date].to_numpy()
        # 只使用有效信号，并要求半衰期在顶部设定的范围内
        valid = np.isfinite(s) & (hl >= hl_min) & (hl <= hl_max)
        # 进入平仓带或信号无效时，把主动方向清零
        direction = np.where((np.abs(s) < s_close) | ~valid, 0, direction)
        # 残差偏低时做多，等它向均值回归
        direction = np.where(valid & (s < -s_open), 1, direction)
        # 残差偏高时做空，等它向均值回归
        direction = np.where(valid & (s > s_open), -1, direction)
        # 扣掉方向在 PCA 因子上的投影，再除以波动率，转成 ETF 对冲权重，这里主要是计算仓位强度
        weight = (direction - loading.T @ (loading @ direction)) / vols.loc[date].to_numpy() # loading.shape = (k, n), direction.shape = (n,)
        # 按绝对权重之和归一化；没有仓位时保持为 0，避免除以 0
        weights.loc[date] = weight / max(np.abs(weight).sum(), 1e-12)

    scores.iloc[window-1:].to_csv(output_dir / f'{phase}_scores.csv', index_label='date')
    weights.iloc[window-1:].to_csv(output_dir / f'{phase}_weights.csv', index_label='date')
    signal_weights = pd.read_csv(output_dir / f'{phase}_weights.csv', parse_dates=['date']).set_index('date')
    # 把前一期信号移到当前收益区间，只统计本段的实际回测日期
    weights = signal_weights.shift(1).loc[start:phase_end].copy()
    asset_returns = np.expm1(returns.loc[weights.index])
    # 第一行填入本日权重，计入从空仓建仓的交易费用
    turnover = weights.diff().fillna(weights).abs().sum(axis=1)
    result = pd.DataFrame(index=weights.index)
    result['gross_ret'] = (weights * asset_returns).sum(axis=1)
    result['turnover'] = turnover
    result['trade_cost'] = turnover * fee
    result['strategy_ret'] = result['gross_ret'] - result['trade_cost']
    result['net_value'] = 1 + result['strategy_ret'].cumsum()
    result.to_csv(output_dir / f'{phase}_backtest.csv', index_label='date')
    backtests[phase] = result
    # 收益区间起点补净值 1，避免过滤掉初始值；保留自然的绘图边距
    initial_date = returns.index.to_series().shift(1).loc[weights.index[0]]
    curves[phase] = pd.DataFrame({
        'PCA strategy': np.r_[1.0, result['net_value']],
        'Equal weight (no costs)': np.r_[1.0, 1 + asset_returns.mean(axis=1).cumsum()]
    }, index=pd.DatetimeIndex([initial_date, *weights.index]))
    ret = result['strategy_ret']
    nav = curves[phase]['PCA strategy']
    metrics[phase] = pd.Series({
        '累计收益': nav.iloc[-1] - 1, '期末净值': nav.iloc[-1],
        '年化夏普': ret.mean() / ret.std(ddof=0) * np.sqrt(252) if ret.std(ddof=0) > 0 else np.nan,
        '最大回撤': (nav / nav.cummax() - 1).min(),
        '累计换手': turnover.sum(), '交易费用': result['trade_cost'].sum(),
        '持仓天数': weights.abs().sum(axis=1).gt(1e-12).sum(), '回测天数': len(ret)
    })
    print(f'{phase}：{weights.index[0].date()}～{weights.index[-1].date()}，{len(ret)} 日', flush=True)

## 3. 训练、验证、测试净值与回撤

每段都使用同一批已选 ETF 的等权无费用基准，基准按日再平衡、固定初始资金加法累计。PCA 多空组合与等权多头基准的市场暴露不同。

仍沿用信号日收盘即时成交的理想化假设；费用按目标权重差估算，不包含价格漂移再平衡、借券、滑点和市场冲击。

上排画净值，下排画对应的百分比回撤。策略使用蓝色，等权基准使用灰色；回撤用同色半透明区域表示。回撤按 `净值 / 历史最高净值 - 1` 计算，包含每段初始净值 1。

In [ ]:
import matplotlib.dates as mdates

# 三列分别对应训练、验证和测试；上排净值，下排回撤
fig, axes = plt.subplots(2, 3, figsize=(19, 8), sharex='col')
for j, (phase, title) in enumerate([('train', 'Training'), ('valid', 'Validation'), ('test', 'Test (out-of-sample)')]):
    phase_curves = curves[phase]
    strategy_nav = phase_curves['PCA strategy']
    baseline_nav = phase_curves['Equal weight (no costs)']
    # 当前是加法累计，基准净值差分就是对应的每日盈亏
    baseline_ret = baseline_nav.diff().dropna()
    baseline_sharpe = baseline_ret.mean() / baseline_ret.std(ddof=0) * np.sqrt(252) if baseline_ret.std(ddof=0) > 0 else np.nan
    strategy_sharpe = metrics[phase]['年化夏普']

    # 净值：策略蓝色，基准灰色，保留初始值 1
    ax = axes[0, j]
    ax.plot(strategy_nav.index, strategy_nav, color='tab:blue', lw=2,
            label=f'PCA stat-arb (long-short, fee = {fee:g})')
    ax.plot(baseline_nav.index, baseline_nav, color='tab:gray', lw=1.5,
            label=f'Baseline: EW {len(codes)} eligible ETFs')
    ax.axhline(1, color='black', lw=0.6)
    ax.set_title(f'{title}: strategy Sharpe {strategy_sharpe:+.2f} vs baseline {baseline_sharpe:+.2f}', fontsize=10)
    ax.set_ylabel('NAV (additive, start = 1)')
    ax.legend(fontsize=8, loc='upper left')
    ax.grid(alpha=0.3)
    ax.tick_params(axis='x', labelbottom=True)

    # 回撤为负值；乘 100 后用百分数刻度展示，越接近 0 越好
    strategy_dd = (strategy_nav / strategy_nav.cummax() - 1) * 100
    baseline_dd = (baseline_nav / baseline_nav.cummax() - 1) * 100
    ax = axes[1, j]
    ax.fill_between(baseline_dd.index, baseline_dd, 0, color='tab:gray', alpha=0.35,
                    label=f'Baseline EW{len(codes)}')
    ax.fill_between(strategy_dd.index, strategy_dd, 0, color='tab:blue', alpha=0.35,
                    label='Strategy')
    ax.axhline(0, color='black', lw=0.6)
    ax.set_ylim(top=0)
    ax.set_title(f'{title}: drawdown (additive NAV)', fontsize=10)
    ax.set_ylabel('Drawdown (%)')
    ax.legend(fontsize=8, loc='lower left')
    ax.grid(alpha=0.3)

    # 每列按自身日期范围显示，保留默认前后边距
    for ax in axes[:, j]:
        ax.xaxis.set_major_locator(mdates.AutoDateLocator(minticks=3, maxticks=5))
        ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
        ax.tick_params(axis='x', labelrotation=25)

fig.suptitle(f'{sector}: PCA stat-arb vs equal-weight sector portfolio', fontsize=14)
plt.tight_layout(rect=[0, 0, 1, 0.95])
plt.show()
# 指标表保持原来的计算口径
df_metrics = pd.DataFrame({'训练': metrics['train'], '验证': metrics['valid'], '测试': metrics['test']})
df_metrics.to_csv(output_dir / 'metrics.csv', index_label='指标', encoding='utf-8-sig')
df_metrics